# VitiGuard MD — обучение моделей диагностики в Google Colab

Ноутбук запускает все эксперименты модуля диагностики (ml/pr-006 — ml/pr-013):

- обучение и сравнение ResNet-50, EfficientNet-B0 и ViT-B/16;
- оценку на тестовой выборке и на полевых снимках HERMOS;
- тепловые карты Grad-CAM;
- маски поражения OpenCV и U-Net (IoU, Dice);
- экспорт лучшей модели в ONNX.

**Перед запуском:**

1. Включите GPU: «Среда выполнения» → «Сменить среду выполнения» → T4 GPU.
2. Положите архив `vitiguard_ml.zip` в папку `vitiguard` на Google Диске (`Мой диск/vitiguard/vitiguard_ml.zip`).
3. «Среда выполнения» → «Выполнить все».

Результаты сохраняются сразу на Диск в `Мой диск/vitiguard/results`. Если сессия оборвётся, запустите ноутбук заново: уже обученные модели пропускаются.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/vitiguard"
RESULTS = f"{DRIVE}/results"

Данные распаковываются на локальный диск Colab: чтение снимков с Google Диска во время обучения очень медленное.

In [ ]:
!rm -rf /content/vitiguard && mkdir -p /content/vitiguard
!unzip -q "{DRIVE}/vitiguard_ml.zip" -d /content/vitiguard
%cd /content/vitiguard/ml
!pip install -q -e . albumentations timm grad-cam segmentation-models-pytorch opencv-python-headless "onnx<1.23"

## 1. Классификация: три модели

Примерно 40–70 минут на T4. Модели, уже обученные в прошлой сессии, пропускаются.

In [ ]:
import json
from pathlib import Path

done = set()
results_file = Path(RESULTS) / "results.json"
if results_file.exists():
    done = set(json.loads(results_file.read_text(encoding="utf-8")).get("classification", {}))
remaining = [m for m in ["efficientnet_b0", "resnet50", "vit_b16"] if m not in done]
print("Уже обучены:", sorted(done) or "нет", "| осталось:", remaining)
if remaining:
    !python -m vitiguard_ml.experiments --out "{RESULTS}" --stages classify --models {" ".join(remaining)}

## 2. Grad-CAM

In [ ]:
!python -m vitiguard_ml.experiments --out "{RESULTS}" --stages explain

## 3. Сегментация поражения (OpenCV + U-Net)

In [ ]:
!python -m vitiguard_ml.experiments --out "{RESULTS}" --stages segment

## 4. Экспорт лучшей модели в ONNX

In [ ]:
!python -m vitiguard_ml.experiments --out "{RESULTS}" --stages export

## Итоги

Сводная таблица и архив для отчёта: `Мой диск/vitiguard/results_report.zip` (метрики и рисунки, без тяжёлых весов).

In [ ]:
results = json.loads(results_file.read_text(encoding="utf-8"))
print(f"{'модель':<16}{'test acc':>9}{'test F1':>9}{'field F1':>9}{'МБ':>7}{'CPU мс':>8}")
for name, r in results["classification"].items():
    print(
        f"{name:<16}{r['test']['accuracy']:>9.3f}{r['test']['macro_f1']:>9.3f}"
        f"{r['field']['macro_f1']:>9.3f}{r['size_mb']:>7}{r['latency_ms']['cpu']:>8}"
    )
print("U-Net:", results.get("segmentation", {}).get("test_vs_opencv_masks"))
print("ONNX:", results.get("export"))

In [ ]:
!cd "{RESULTS}" && rm -f ../results_report.zip && zip -qr ../results_report.zip results.json figures
from IPython.display import Image, display

display(Image(f"{RESULTS}/figures/training_curves.png"))
display(Image(f"{RESULTS}/figures/gradcam.png", width=700))